In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import itertools
import scipy 
import math
import matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64)
pi = torch.tensor(np.pi,dtype=torch.float64)

from utils import PiecewiseGQ2D_weights_points, SquareMesh2D_points, split_domain_boundary
from utils import s2_uniform_grid_init
from utils import output_convergence_order_l2
from utils import plot_err_convergence

In [2]:
class ShallowReLUkNetwork:
    def __init__(self, ws, bs, train_pts, k=1, m1=1, m2=1):
        self.n = ws.shape[0]
        self.W = ws 
        self.b = bs
        self.k = k
        self.m1 = m1 
        self.m2 = m2
        pts_dom, pts_bd = split_domain_boundary(train_pts)
        self.pts_dom = pts_dom.requires_grad_(True)
        self.pts_bd = pts_bd
    
    def feat_extract(self, x):
        return F.relu(x @ self.W.t() + self.b) ** self.k

    def target(self, x):
        z = torch.sin(self.m1 * pi*x[:,0:1])*torch.sin(self.m2 * pi*x[:,1:2])
        return z 
    
    def rhs(self, x):
        u = torch.sin(self.m1 * pi * x[:, 0:1]) * torch.sin(self.m2 * pi * x[:, 1:2])
        coeff = 1 - (self.m1 * pi) ** 2 - (self.m2 * pi) ** 2
        return coeff * u
    
    def lapfeat(self, features, x):
        lap = []
        for i in range(features.shape[1]):  # loop over feature dimension m
            g = features[:, i:i+1]          # (N,1)

            # gradient wrt x
            grad_g = torch.autograd.grad(
                g, x, torch.ones_like(g), create_graph=True
            )[0]   # (N,2)

            # second derivatives
            lap_g = 0
            for d in range(x.shape[1]):  # loop over x,y
                grad2 = torch.autograd.grad(
                    grad_g[:, d], x, torch.ones_like(grad_g[:, d]), create_graph=True
                )[0][:, d:d+1].detach()
                lap_g = lap_g + grad2

            lap.append(lap_g)
        
        return torch.cat(lap, dim=1)   # (N,m)


    def assemble(self):
        gs = self.feat_extract(self.pts_dom)
        lap_gs = self.lapfeat(gs, self.pts_dom)
        A_dom = lap_gs + gs
        b_dom = self.rhs(self.pts_dom)
        A_bd = self.feat_extract(self.pts_bd)
        b_bd = self.target(self.pts_bd)
        A = torch.concat([A_dom, A_bd])
        b = torch.concat([b_dom, b_bd])       
        return A, b

    def solve(self):
        A, b = self.assemble()

        c = 100.0 
        for i in range(len(A)):
            max_a = abs(A[i,:]).max()
            max_b = A[i,:].max()
            if max_a != max_b: 
                ratio = -c/max_a
                A[i,:] = A[i,:]*ratio
                b[i] = b[i]*ratio
            else: 
                ratio = c/max_a
                A[i,:] = A[i,:]*ratio
                b[i] = b[i]*ratio
        alpha = scipy.linalg.lstsq(A.detach().numpy(),b.detach().numpy())[0]
        self.alpha = torch.tensor(alpha)
        
    def forward(self, x):
        return self.feat_extract(x) @ self.alpha
    
    def eval(self, x):
        y_ref = self.target(x)
        y_pred = self.forward(x)
        rel_errl2 = (y_pred - y_ref).norm() / y_ref.norm()
        return rel_errl2.item()

In [3]:
x_train = SquareMesh2D_points(0,1,100)
x_test = SquareMesh2D_points(0,1,150)
k = 2
m1 = 1
m2 = 1
neuron_num_arr= np.array([2**i for i in range(6,11)]) 
neuron_arr = []
rel_errl2_arr = []
for i, n in enumerate(neuron_num_arr):
    ws, bs = s2_uniform_grid_init(n, k=k)
    model = ShallowReLUkNetwork(
        ws, bs, x_train, k=k, m1=2, m2=m2)
    model.solve()
    rel_errl2 = model.eval(x_test)
    neuron_arr.append(model.n)
    rel_errl2_arr.append(rel_errl2)
    print('num neuron : {:} - rel l2 : {:.4e}'.format(model.n, rel_errl2))
title = "L2 fitting. ReLU$^{}$. $\sin(m_1\pi x_1) \sin(m_2\pi x_2)$, $m_1,m_2 = {}, {}$".format(k, m1, m2)

num neuron : 34 - rel l2 : 3.3066e+00
num neuron : 60 - rel l2 : 1.6943e+00
num neuron : 116 - rel l2 : 2.9501e-01
num neuron : 230 - rel l2 : 1.5128e-01
num neuron : 448 - rel l2 : 1.1639e-01
